In [ ]:
from functools import lru_cache, cache


# -------------------------------------------------------------
# 1) BASIC FUNCTIONS (LeetCode style - class ke andar method)
# -------------------------------------------------------------
class Solution:
    def twoSum(self, nums, target):
        # LeetCode par tumhara solution hamesha ek class ke andar hota hai
        ...

    def helper(self, arr, index):
        # helper functions bana lo, code clean rehta hai
        ...


# -------------------------------------------------------------
# 2) NESTED FUNCTIONS (function ke andar function) - backtracking mein bahut common
# -------------------------------------------------------------
class Solution2:
    def permute(self, nums):
        results = []

        def backtrack(path):
            # "backtrack" function bahar wale "results" aur "nums" ko
            # seedha use kar sakta hai - isko CLOSURE kehte hain.
            # Isse extra parameters pass karne ki zaroorat nahi padti.
            if len(path) == len(nums):
                results.append(path[:])   # [:] se COPY banao, warna bug aayega!
                return
            for num in nums:
                if num not in path:
                    path.append(num)
                    backtrack(path)
                    path.pop()   # explore karne ke baad wapas nikaal do (undo)

        backtrack([])
        return results

# WHY path[:] zaroori hai?
# Agar tum results.append(path) likhoge (bina copy ke), to results ke andar
# saari entries EK HI list ko point karengi. Jab backtracking khatam hoga aur
# path khaali ho jaayega, to results ki saari entries bhi khaali dikhengi.
# Yeh backtracking mein sabse common bug hai - HAMESHA copy banao.


# -------------------------------------------------------------
# 3) PASS-BY-OBJECT-REFERENCE (Python function ko value kaise deta hai)
# -------------------------------------------------------------
# RULE:
#   - MUTABLE cheezein (list, dict, set) -> function ke andar change karo to
#     bahar bhi change dikhega (kyunki same object hai).
#   - IMMUTABLE cheezein (int, str, tuple, bool) -> function ke andar "change"
#     karne se ek NAYA object banta hai, bahar wala variable waisa hi rehta hai.

def modify_list(arr):
    arr.append(99)     # original list modify ho gayi
    arr[0] = -1         # yeh bhi original modify karta hai

def modify_int(x):
    x += 1              # naya int object banta hai, purana x jo bahar hai wo waisa hi rahega

nums = [1, 2, 3]
modify_list(nums)
print(nums)   # [-1, 2, 3, 99]  -> CHANGED

count = 5
modify_int(count)
print(count)  # 5  -> UNCHANGED

# Ismein backtracking ka pattern samjho:
def backtrack(path, results):
    if True:  # is_complete(path) jaisa kuch condition
        results.append(path[:])  # COPY zaroor karo
        return
    # for choice in choices:
    #     path.append(choice)          # shared list modify ki
    #     backtrack(path, results)     # aage recursion mein bheja
    #     path.pop()                   # wapas undo kiya


# -------------------------------------------------------------
# 4) MUTABLE DEFAULT ARGUMENT - EK BADA PYTHON TRAP
# -------------------------------------------------------------
# GALAT tarika:
def add_to_wrong(element, target=[]):
    # yeh default list []  SIRF EK BAAR banti hai, jab function define hota hai.
    # Har call mein WAHI list reuse hoti hai - is se purana data reh jaata hai!
    target.append(element)
    return target

# add_to_wrong(1)  -> [1]
# add_to_wrong(2)  -> [1, 2]   <-- expect kiya tha [2], par mila [1, 2]!

# SAHI tarika: default None rakho, andar naya list banao
def add_to_correct(element, target=None):
    if target is None:
        target = []
    target.append(element)
    return target


# -------------------------------------------------------------
# 5) NONLOCAL - nested function se outer variable modify karna (int ke liye)
# -------------------------------------------------------------
def count_nodes(root):
    count = 0

    def dfs(node):
        nonlocal count   # zaroori hai, warna "count += 1" ek NAYA local variable bana dega
        if not node:
            return
        count += 1
        dfs(node.left)
        dfs(node.right)

    dfs(root)
    return count

# Alternative: nonlocal na chahiye ho to ek list [0] use kar sakte ho (thoda kam clean)


# -------------------------------------------------------------
# 6) RECURSION, CALL STACK, AUR sys.setrecursionlimit
# -------------------------------------------------------------
import sys
print(sys.getrecursionlimit())   # default: 1000

def factorial(n):
    if n <= 1:
        return 1
    return n * factorial(n - 1)

# factorial(5000)  -> yeh RecursionError dega kyunki depth 1000 se zyada hai!

sys.setrecursionlimit(10**6)   # isse depth 10 lakh tak badha do
# Ab bade graph/tree pe DFS chala sakte ho bina crash ke.

# KAB DARNA CHAHIYE (depth ka andaza):
#   balanced binary tree (n nodes)   -> depth O(log n)  -> SAFE hai
#   linked list / skewed tree        -> depth O(n)      -> RISKY agar n > 1000
#   backtracking (chhote choices)    -> depth chhoti     -> usually SAFE
#   graph DFS (n nodes)              -> depth O(n)       -> recursionlimit badhao

# Agar recursion depth bahut zyada ho sakti hai, to RECURSION ko ITERATION
# mein convert kar do (apna khud ka stack banao):
def dfs_recursive(node, visited, graph):
    visited.add(node)
    for neighbor in graph[node]:
        if neighbor not in visited:
            dfs_recursive(neighbor, visited, graph)

def dfs_iterative(start, visited, graph):
    stack = [start]
    while stack:
        node = stack.pop()
        if node in visited:
            continue
        visited.add(node)
        for neighbor in graph[node]:
            if neighbor not in visited:
                stack.append(neighbor)
# NOTE: Python tail recursion ko optimize NAHI karta (kuch languages karte hain).
# Matlab har recursive call, chahe wo "tail" pe ho ya na ho, call stack mein
# ek frame add karti hai.


# -------------------------------------------------------------
# 7) MEMOIZATION - @cache aur @lru_cache (DP ke liye shortcut)
# -------------------------------------------------------------
@cache   # Python 3.9+  (same as @lru_cache(maxsize=None))
def fib(n):
    if n <= 1:
        return n
    return fib(n - 1) + fib(n - 2)
# Bina @cache ke, fib(n) exponential time leta hai. @cache lagate hi
# Python automatically already-calculated answers yaad rakh leta hai (memoization),
# aur function fast ho jaata hai (O(n) tak).

# 2D DP example (Longest Common Subsequence):
def make_lcs(s, t):
    @cache
    def lcs(i, j):
        if i < 0 or j < 0:
            return 0
        if s[i] == t[j]:
            return 1 + lcs(i - 1, j - 1)
        return max(lcs(i - 1, j), lcs(i, j - 1))
    return lcs(len(s) - 1, len(t) - 1)

# LeetCode class ke andar use karna (nested function trick):
class SolutionDP:
    def coinChange(self, coins, amount):
        @cache
        def dp(remaining):
            if remaining == 0:
                return 0
            if remaining < 0:
                return float('inf')
            return 1 + min(dp(remaining - c) for c in coins)
            # 'coins' outer scope se seedha mil jaata hai (closure), parameter
            # banane ki zaroorat nahi - isse cache key bhi saaf rehti hai.

        result = dp(amount)
        return result if result != float('inf') else -1

# ZAROORI RULE: @cache lagane wale function ke ARGUMENTS hashable hone chahiye.
# List nahi chalegi, TUPLE use karo:
# WRONG: dp(index, remaining_items)   # agar remaining_items list hai -> error
# RIGHT: dp(index, tuple(remaining_items))

# Agar multiple test cases run kar rahe ho (local testing), to cache clear karo:
# fib.cache_clear()

# KAB manual memo (dictionary) use karo, @cache nahi:
#   - jab mutable state pass karna ho jo hashable nahi ban sakta
#   - jab cache ko beech mein dekhna/badalna ho
#   - jab state space bahut bada ho aur memory control chahiye
#   - jab recursion bahut deep ho (decorator thoda overhead deta hai)


# -------------------------------------------------------------
# 8) OOP BASICS - DSA mein jo Node classes baar baar aati hain
# -------------------------------------------------------------
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

class TrieNode:
    def __init__(self):
        self.children = {}      # ya [None] * 26 agar sirf lowercase letters hain
        self.is_end = False

class GraphNode:
    def __init__(self, val=0, neighbors=None):
        self.val = val
        self.neighbors = neighbors if neighbors is not None else []


# CUSTOM COMPARISON for heap (agar apna khud ka object heap mein daalna hai):
class ListNodeComparable:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

    def __lt__(self, other):
        # __lt__ = "less than" -> heapq isi function se compare karta hai
        return self.val < other.val

# Alternative (zyada preferred): object ke saath ek tie-breaker counter tuple
# mein daal do heap mein, taaki class modify hi na karni pade. Yeh Heap wali
# file (05) mein detail se dikhaya gaya hai.